In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import tensorflow as tf

data_dir = 'Rice_Image_Dataset'
batch_size = 256
img_height = 224
img_width = 224

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=0,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=0,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

Found 75000 files belonging to 5 classes.
Using 60000 files for training.
Found 75000 files belonging to 5 classes.
Using 15000 files for validation.


## Transfer Learning Model

In [3]:
base_model = tf.keras.applications.MobileNetV2(input_shape=(img_height, img_width, 3),
                                               include_top=False,
                                               weights='imagenet')
base_model.trainable = False  # Freeze the pre-trained model

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(5, activation='softmax')  # 5 classes for the rice types
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [4]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10


2025-03-08 16:20:58.613894: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_4' with dtype int32 and shape [60000]
	 [[{{node Placeholder/_4}}]]
2025-03-08 16:20:58.614042: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_4' with dtype int32 and shape [60000]
	 [[{{node Placeholder/_4}}]]
2025-03-08 16:20:59.248301: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


235/235 [==============================] - ETA: 0s - loss: 0.1976 - accuracy: 0.9318

2025-03-08 16:25:56.900746: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [15000]
	 [[{{node Placeholder/_0}}]]
2025-03-08 16:25:56.900908: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [15000]
	 [[{{node Placeholder/_0}}]]


235/235 [==============================] - 382s 2s/step - loss: 0.1976 - accuracy: 0.9318 - val_loss: 0.0804 - val_accuracy: 0.9757
Epoch 2/10
235/235 [==============================] - 387s 2s/step - loss: 0.0770 - accuracy: 0.9736 - val_loss: 0.0855 - val_accuracy: 0.9710
Epoch 3/10
235/235 [==============================] - 374s 2s/step - loss: 0.0678 - accuracy: 0.9766 - val_loss: 0.0534 - val_accuracy: 0.9815
Epoch 4/10
235/235 [==============================] - 391s 2s/step - loss: 0.0569 - accuracy: 0.9808 - val_loss: 0.0497 - val_accuracy: 0.9824
Epoch 5/10
235/235 [==============================] - 396s 2s/step - loss: 0.0549 - accuracy: 0.9811 - val_loss: 0.0511 - val_accuracy: 0.9821
Epoch 6/10
235/235 [==============================] - 388s 2s/step - loss: 0.0466 - accuracy: 0.9837 - val_loss: 0.0494 - val_accuracy: 0.9838
Epoch 7/10
235/235 [==============================] - 377s 2s/step - loss: 0.0482 - accuracy: 0.9829 - val_loss: 0.0465 - val_accuracy: 0.9839
Epoch 8/10

## CNN Base Model

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Define image dimensions and batch size (adjust as needed)
img_height = 224
img_width = 224
batch_size = 256

# Build a baseline CNN model
model = Sequential([
    # Convolutional block 1
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D((2, 2)),
    
    # Convolutional block 2
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    # Convolutional block 3
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    # Classification head
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),  # Helps reduce overfitting
    Dense(5, activation='softmax')  # 5 output classes for your rice varieties
])

# Compile the model with an appropriate optimizer and loss function
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Set up an EarlyStopping callback to stop training if validation loss doesn't improve
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model using the training and validation datasets
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,  # Adjust the number of epochs as needed
    callbacks=[early_stop]
)

# Print the model summary for a quick review of the architecture
model.summary()


Epoch 1/10
235/235 [==============================] - 950s 4s/step - loss: 3.2252 - accuracy: 0.9081 - val_loss: 0.0840 - val_accuracy: 0.9722
Epoch 2/10
235/235 [==============================] - 958s 4s/step - loss: 0.1388 - accuracy: 0.9558 - val_loss: 0.0758 - val_accuracy: 0.9745
Epoch 3/10
235/235 [==============================] - 931s 4s/step - loss: 0.1103 - accuracy: 0.9650 - val_loss: 0.0660 - val_accuracy: 0.9795
Epoch 4/10
235/235 [==============================] - 867s 4s/step - loss: 0.0790 - accuracy: 0.9756 - val_loss: 0.0383 - val_accuracy: 0.9892
Epoch 5/10
235/235 [==============================] - 851s 4s/step - loss: 0.0721 - accuracy: 0.9769 - val_loss: 0.0806 - val_accuracy: 0.9740
Epoch 6/10
235/235 [==============================] - 849s 4s/step - loss: 0.0602 - accuracy: 0.9814 - val_loss: 0.0424 - val_accuracy: 0.9894
Epoch 7/10
235/235 [==============================] - 852s 4s/step - loss: 0.0449 - accuracy: 0.9858 - val_loss: 0.0379 - val_accuracy: 0.9892